# Score, Repack, and Minimize a Pose

This tutorial walks through a compact refinement workflow using TMol's native
`PoseStack`, score function, packer, minimizer, and py3Dmol viewer. The goal is
to make each stage explicit: load a pose, score it, repack side chains, minimize
Cartesian coordinates, score the refined pose again, and inspect the final model.

The notebook uses the first ten residues of ubiquitin from TMol's checked-in test
fixtures. That keeps the tutorial fast while still running the same APIs used for
larger protein workflows.

## Imports and Input Pose

TMol stores molecular systems in a `PoseStack`. Here we load a small residue
range from a PDB file and construct the default `beta2016` score function on the
same device. The score function and the packer share the same parameter database.

In [1]:
from pathlib import Path

import torch

import tmol
from tmol.database import ParameterDatabase
from tmol.io import pose_stack_from_pdb
from tmol.optimization.minimizers import run_cart_min
from tmol.pack.pack_rotamers import pack_rotamers
from tmol.pack.packer_task import PackerPalette, PackerTask
from tmol.pack.rotamer.dunbrack.dunbrack_chi_sampler import (
    create_dunbrack_sampler_from_database,
)
from tmol.pack.rotamer.fixed_aa_chi_sampler import FixedAAChiSampler
from tmol.pack.rotamer.include_current_sampler import IncludeCurrentSampler
from tmol.score import beta2016_score_function

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
repo_root = Path(tmol.__file__).resolve().parents[1]
pdb_path = repo_root / "tmol" / "tests" / "data" / "pdb" / "1ubq.pdb"

param_db = ParameterDatabase.get_default()
pose_stack = pose_stack_from_pdb(
    str(pdb_path),
    device,
    residue_start=0,
    residue_end=10,
)
sfxn = beta2016_score_function(device, param_db=param_db)

print(f"Loaded {pose_stack.n_poses} pose with {pose_stack.max_n_blocks} residues")

Loaded 1 pose with 10 residues


## Score the Starting Coordinates

Rendering a whole-pose scoring module binds the score function to this pose layout.
Calling it with the pose coordinates returns one weighted total score per pose in
the stack. Lower scores are better for this score function.

In [2]:
def total_score(pose_stack, sfxn):
    scorer = sfxn.render_whole_pose_scoring_module(pose_stack)
    score = scorer(pose_stack.coords).detach().cpu()
    return float(score[0])

start_score = total_score(pose_stack, sfxn)
print(f"starting beta2016 score: {start_score:.3f}")

starting beta2016 score: 33.929


## Build a Repacking Task

Repacking keeps the amino-acid sequence fixed and searches side-chain conformers.
A `PackerTask` defines which residues may change conformation and which rotamer
samplers are allowed to propose alternatives. This tutorial uses Dunbrack rotamers,
fixed amino-acid chi samples, and the current input conformation.

In [3]:
task = PackerTask(pose_stack, PackerPalette())
task.restrict_to_repacking()
task.add_conformer_sampler(create_dunbrack_sampler_from_database(param_db, device))
task.add_conformer_sampler(FixedAAChiSampler())
task.add_conformer_sampler(IncludeCurrentSampler())

packed_pose_stack = pack_rotamers(pose_stack, sfxn, task)
packed_score = total_score(packed_pose_stack, sfxn)
print(f"score after repacking: {packed_score:.3f}")

score after repacking: 11.909


## Minimize Cartesian Coordinates

Repacking changes discrete side-chain conformers. Cartesian minimization then
optimizes continuous atom coordinates against the same score function. The tutorial
bounds the minimizer iteration count so the example remains lightweight; production
refinement can use the default optimizer settings or a tighter coordinate mask.

In [4]:
minimized_pose_stack = run_cart_min(
    packed_pose_stack,
    sfxn,
    optimizer_kwargs={"max_iter": 20, "gradtol": 0.5},
)
final_score = total_score(minimized_pose_stack, sfxn)
print(f"score after repack + minimize: {final_score:.3f}")

score after repack + minimize: 0.634


## Compare Energies

The absolute score depends on the selected score function and residue range. For a
refinement workflow, the useful check is that each stage is scored with the same
`ScoreFunction` and parameter database.

In [5]:
print("stage                       beta2016")
print(f"starting pose             {start_score:9.3f}")
print(f"after repacking           {packed_score:9.3f}")
print(f"after repack + minimize   {final_score:9.3f}")

stage                       beta2016
starting pose                33.929
after repacking              11.909
after repack + minimize       0.634


## View the Refined Pose

`tmol.view()` returns a real `py3Dmol.view` object. Because this notebook stores
the rendered output, the built documentation publishes the same draggable viewer
without rerunning TMol during the Sphinx build.

In [6]:
def pose_center(pose_stack):
    coords = pose_stack.coords[0].detach().cpu()
    finite = coords[torch.isfinite(coords).all(dim=-1)]
    center = finite.mean(dim=0)
    return {"x": float(center[0]), "y": float(center[1]), "z": float(center[2])}

viewer = tmol.view(minimized_pose_stack, width=720, height=420)
viewer.addLabel(
    f"beta2016: {start_score:.2f} -> {final_score:.2f}",
    {
        "position": pose_center(minimized_pose_stack),
        "backgroundColor": "white",
        "fontColor": "black",
        "fontSize": 14,
        "inFront": True,
        "showBackground": True,
    },
)
viewer.zoomTo()
viewer